# Diagnóstico de calidad de datos (GA1-220501094-01-AA1-EV06)

**Proyecto:** Análisis y Predicción de Cancelación de Suscripciones (Churn Rate) en Plataformas Digitales
**Aprendiz:** Nicolás Muñoz Hernández · Ficha 3223873 · CTMA – SENA Antioquia

**Pregunta analítica:** ¿Qué patrones de interacción, frecuencia de uso y nivel de soporte técnico en los primeros 30 días permiten predecir el riesgo de cancelación?

## Momento 1. Datasets seleccionados

| Dataset | Archivo | Descripción | Relación con el problema |
|---|---|---|---|
| Suscripción (SaaS, sintético) | `customer_subscription_churn_usage_patterns.csv` | Plan, tarifa, horas de uso, tickets de soporte, fallos de pago, antigüedad y churn | Permite explorar uso, soporte y pagos frente a la cancelación |
| Telco (telecomunicaciones) | `churn.csv` | Planes, minutos y cargos por franja, llamadas a servicio al cliente y churn | Permite analizar si el uso y las llamadas a soporte se asocian a la cancelación |

**Aspectos de calidad a revisar:** valores faltantes, tipos de datos, duplicados, rangos incoherentes, balance de la variable objetivo (`churn`) y redundancia entre variables.

> **Nota de corrección:** el informe EV03 indicaba 10.000 registros para el Dataset 1. Al cargar el archivo real se comprueba que tiene **2.800 filas y 10 columnas**.

## Momento 2. Carga de datos

In [18]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df_suscripcion = pd.read_csv("../data/raw/customer_subscription_churn_usage_patterns.csv")
df_telco = pd.read_csv("../data/raw/churn.csv")

print("df_suscripcion:", df_suscripcion.shape)
print("df_telco:      ", df_telco.shape)

df_suscripcion: (2800, 10)
df_telco:       (4250, 21)


## Momento 3. Inspección de la estructura

In [19]:
def inspeccionar(df, nombre):
    print("=" * 70)
    print(f"DATASET: {nombre}")
    print("=" * 70)
    print("\n--- head() ---");  display(df.head())
    print("\n--- tail() ---");  display(df.tail())
    print("\n--- shape ---");   print(df.shape)
    print("\n--- columns ---"); print(df.columns.tolist())
    print("\n--- dtypes ---");  print(df.dtypes)
    print("\n--- info() ---");  df.info()
    print("\n--- describe() ---"); display(df.describe())
    print("\n--- describe(include='object') ---")
    try:
        display(df.describe(include="object"))
    except ValueError:
        print("No hay columnas de tipo object")

### 3.1 Dataset de suscripción

In [20]:
inspeccionar(df_suscripcion, "df_suscripcion")

DATASET: df_suscripcion

--- head() ---


,user_id,signup_date,plan_type,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,churn
0,1,2023-04-15,Premium,699,1.1,4,1,8,14,Yes
1,2,2023-08-27,Premium,699,2.6,6,0,35,1,Yes
2,3,2023-10-12,Premium,699,14.3,8,3,2,14,Yes
3,4,2023-12-11,Basic,199,17.6,5,2,11,9,Yes
4,5,2023-02-14,Basic,199,9.8,5,2,6,38,Yes



--- tail() ---


,user_id,signup_date,plan_type,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,churn
2795,2796,2023-03-10,Standard,399,6.7,1,1,21,59,No
2796,2797,2024-11-20,Standard,399,11.0,4,0,6,4,No
2797,2798,2024-08-13,Standard,399,11.0,0,4,22,59,No
2798,2799,2024-05-23,Premium,699,15.0,3,4,25,20,No
2799,2800,2024-03-21,Standard,399,2.0,0,1,12,1,Yes



--- shape ---
(2800, 10)

--- columns ---
['user_id', 'signup_date', 'plan_type', 'monthly_fee', 'avg_weekly_usage_hours', 'support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago', 'churn']

--- dtypes ---
user_id                     int64
signup_date                   str
plan_type                     str
monthly_fee                 int64
avg_weekly_usage_hours    float64
support_tickets             int64
payment_failures            int64
tenure_months               int64
last_login_days_ago         int64
churn                         str
dtype: object

--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   user_id                 2800 non-null   int64  
 1   signup_date             2800 non-null   str    
 2   plan_type               2800 non-null   str    
 3   monthly_fee             2800 non-nul

,user_id,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago
count,2800.000000,2800.000000,2800.000000,2800.000000,2800.000000,2800.000000,2800.000000
mean,1400.500000,434.214286,12.891429,3.887857,2.491786,18.612857,30.005000
std,808.434702,205.678472,7.109691,2.606419,1.691647,10.374487,17.852757
min,1.000000,199.000000,0.500000,0.000000,0.000000,1.000000,0.000000
25%,700.750000,199.000000,6.700000,2.000000,1.000000,10.000000,14.000000
50%,1400.500000,399.000000,12.800000,4.000000,2.000000,18.000000,30.000000
75%,2100.250000,699.000000,19.200000,6.000000,4.000000,27.000000,46.000000
max,2800.000000,699.000000,25.000000,8.000000,5.000000,36.000000,60.000000



--- describe(include='object') ---


C:\Users\IK\AppData\Local\Temp\ipykernel_7432\3385040580.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include="object"))


,signup_date,plan_type,churn
count,2800,2800,2800
unique,710,3,2
top,2024-05-02,Premium,Yes
freq,10,944,1605


**Interpretación (suscripción):** 2.800 clientes y 10 variables. `signup_date`, `plan_type` y `churn` son texto. `monthly_fee` toma solo tres valores según el plan. Los rangos de uso (0,5–25 h), tickets (0–8), fallos de pago (0–5) y antigüedad (1–36 meses) son plausibles.

### 3.2 Dataset Telco

In [21]:
inspeccionar(df_telco, "df_telco")

DATASET: df_telco

--- head() ---


,customer_id,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls,churn
0,932aff1e-186f-418c-8716-28bba167c384,NY,86,area_code_510,False,False,0,70.7,125,12.02,211.0,113,17.94,174.6,107,7.86,0.0,0,0.0,2,False
1,143a0b05-b95a-4d00-ad2a-9c1b18234c29,VT,165,area_code_510,False,False,0,242.9,126,41.29,209.8,65,17.83,228.4,126,10.28,0.0,0,0.0,1,False
2,45debf35-7e4d-42da-ba4e-d737a8997e16,KS,70,area_code_415,False,False,0,222.8,114,37.88,215.9,113,18.35,223.5,122,10.06,0.0,0,0.0,1,False
3,76b4b68f-1113-4411-88ad-de38b482e9fb,NM,158,area_code_408,False,False,0,172.4,114,29.31,256.6,69,21.81,235.3,104,10.59,0.0,0,0.0,2,False
4,6ee9cecd-1a09-4f4a-937e-aa848c392bed,ND,101,area_code_415,False,False,0,217.7,118,37.01,231.7,128,19.69,185.3,128,8.34,0.0,0,0.0,3,False



--- tail() ---


,customer_id,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls,churn
4245,e1db5067-6484-4a5d-b68d-ab869fff027d,UT,36,area_code_415,False,True,16,149.4,111,25.40,131.8,113,11.20,132.7,87,5.97,6.7,2,1.81,0,False
4246,99ad8c1e-c90b-44d5-ad36-d4d87aae0d85,MN,24,area_code_510,False,True,25,164.9,110,28.03,209.3,105,17.79,231.2,55,10.40,6.7,9,1.81,1,False
4247,b2287275-f563-40ae-ad9c-06844ca539f4,DE,97,area_code_415,False,True,32,183.4,94,31.18,269.1,120,22.87,203.5,38,9.16,6.7,4,1.81,5,False
4248,9908263a-a9c9-48f5-9c26-ca7cc9bd5917,IN,81,area_code_408,False,True,46,168.3,124,28.61,270.9,103,23.03,222.5,98,10.01,6.7,2,1.81,4,False
4249,ab236f86-ad9b-461e-9193-3176d3925a1f,VA,133,area_code_408,False,True,39,239.9,107,40.78,253.8,77,21.57,128.7,85,5.79,6.7,3,1.81,5,False



--- shape ---
(4250, 21)

--- columns ---
['customer_id', 'state', 'account_length', 'area_code', 'international_plan', 'voice_mail_plan', 'number_vmail_messages', 'total_day_minutes', 'total_day_calls', 'total_day_charge', 'total_eve_minutes', 'total_eve_calls', 'total_eve_charge', 'total_night_minutes', 'total_night_calls', 'total_night_charge', 'total_intl_minutes', 'total_intl_calls', 'total_intl_charge', 'number_customer_service_calls', 'churn']

--- dtypes ---
customer_id                          str
state                                str
account_length                     int64
area_code                            str
international_plan                  bool
voice_mail_plan                     bool
number_vmail_messages              int64
total_day_minutes                float64
total_day_calls                    int64
total_day_charge                 float64
total_eve_minutes                float64
total_eve_calls                    int64
total_eve_charge                 flo

,account_length,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls
count,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000,4250.000000
mean,100.236235,7.631765,180.259600,99.907294,30.644682,200.173906,100.176471,17.015012,200.527882,99.839529,9.023892,10.256071,4.426353,2.769654,1.559059
std,39.698401,13.439882,54.012373,19.850817,9.182096,50.249518,19.908591,4.271212,50.353548,20.093220,2.265922,2.760102,2.463069,0.745204,1.311434
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,73.000000,0.000000,143.325000,87.000000,24.365000,165.925000,87.000000,14.102500,167.225000,86.000000,7.522500,8.500000,3.000000,2.300000,1.000000
50%,100.000000,0.000000,180.450000,100.000000,30.680000,200.700000,100.000000,17.060000,200.450000,100.000000,9.020000,10.300000,4.000000,2.780000,1.000000
75%,127.000000,16.000000,216.200000,113.000000,36.750000,233.775000,114.000000,19.867500,234.700000,113.000000,10.560000,12.000000,6.000000,3.240000,2.000000
max,243.000000,52.000000,351.500000,165.000000,59.760000,359.300000,170.000000,30.540000,395.000000,175.000000,17.770000,20.000000,20.000000,5.400000,9.000000



--- describe(include='object') ---


C:\Users\IK\AppData\Local\Temp\ipykernel_7432\3385040580.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include="object"))


,customer_id,state,area_code
count,4250,4250,4250
unique,4250,51,3
top,932aff1e-186f-418c-8716-28bba167c384,WV,area_code_415
freq,1,139,2108


**Interpretación (Telco):** 4.250 clientes y 21 variables. Hay 3 columnas booleanas (`international_plan`, `voice_mail_plan`, `churn`) y 3 de texto (`customer_id`, `state`, `area_code`). Las franjas diurna, vespertina, nocturna e internacional traen minutos, llamadas y cargos.

### 3.3 Tabla de diagnóstico estructural (generada con código)

In [22]:
def resumen_estructural(df, nombre):
    tipos = df.dtypes.astype(str).value_counts().to_dict()
    return {"Dataset": nombre, "Filas": df.shape[0], "Columnas": df.shape[1],
            "Tipos de datos": tipos}

tabla_estructural = pd.DataFrame([
    resumen_estructural(df_suscripcion, "df_suscripcion"),
    resumen_estructural(df_telco, "df_telco"),
])
tabla_estructural

,Dataset,Filas,Columnas,Tipos de datos
0,df_suscripcion,2800,10,"{'int64': 6, 'str': 3, 'float64': 1}"
1,df_telco,4250,21,"{'float64': 8, 'int64': 7, 'str': 3, 'bool': 3}"


## Momento 4. Diagnóstico de calidad

### 4.1 Valores faltantes y duplicados

In [23]:
for nombre, df in [("df_suscripcion", df_suscripcion), ("df_telco", df_telco)]:
    print(f"{nombre}: nulos totales = {df.isna().sum().sum()} | filas duplicadas = {df.duplicated().sum()}")

print("\nNulos por columna (suscripción):"); print(df_suscripcion.isna().sum())
print("\nNulos por columna (telco):");       print(df_telco.isna().sum())

df_suscripcion: nulos totales = 0 | filas duplicadas = 0
df_telco: nulos totales = 0 | filas duplicadas = 0

Nulos por columna (suscripción):
user_id                   0
signup_date               0
plan_type                 0
monthly_fee               0
avg_weekly_usage_hours    0
support_tickets           0
payment_failures          0
tenure_months             0
last_login_days_ago       0
churn                     0
dtype: int64

Nulos por columna (telco):
customer_id                      0
state                            0
account_length                   0
area_code                        0
international_plan               0
voice_mail_plan                  0
number_vmail_messages            0
total_day_minutes                0
total_day_calls                  0
total_day_charge                 0
total_eve_minutes                0
total_eve_calls                  0
total_eve_charge                 0
total_night_minutes              0
total_night_calls                0
total_night_

**Interpretación:** ninguno de los dos datasets tiene valores nulos ni filas duplicadas, por lo que no hace falta imputar ni eliminar registros.

### 4.2 Dataset de suscripción: tipos, categorías y consistencia

In [24]:
# Identificador único
print("user_id único:", df_suscripcion["user_id"].is_unique)

# Variable objetivo
print("\nDistribución de churn:")
print(df_suscripcion["churn"].value_counts())
print(df_suscripcion["churn"].value_counts(normalize=True).round(3))

# Fechas: texto -> datetime
fechas = pd.to_datetime(df_suscripcion["signup_date"], errors="coerce")
print("\nFechas no convertibles:", fechas.isna().sum())
print("Rango de fechas:", fechas.min().date(), "->", fechas.max().date())

# Plan vs tarifa
print("\nPlan vs tarifa:")
print(df_suscripcion.groupby("plan_type")["monthly_fee"].unique())

# Rangos
columnas_num = ["avg_weekly_usage_hours", "support_tickets", "payment_failures",
                "tenure_months", "last_login_days_ago"]
print("\nMínimos y máximos:")
print(df_suscripcion[columnas_num].agg(["min", "max"]))

# Coherencia entre fecha de registro y antigüedad
# (fecha de referencia supuesta: la fecha de hoy; el dataset no indica fecha de corte)
meses_estimados = (pd.Timestamp.today().normalize() - fechas).dt.days / 30.4
print("\nCorrelación tenure_months vs meses desde signup_date:",
      round(df_suscripcion["tenure_months"].corr(meses_estimados), 3))

user_id único: True

Distribución de churn:
churn
Yes    1605
No     1195
Name: count, dtype: int64
churn
Yes    0.573
No     0.427
Name: proportion, dtype: float64

Fechas no convertibles: 0
Rango de fechas: 2023-01-01 -> 2024-12-31

Plan vs tarifa:
plan_type
Basic       [199]
Premium     [699]
Standard    [399]
Name: monthly_fee, dtype: object

Mínimos y máximos:
     avg_weekly_usage_hours  support_tickets  payment_failures  tenure_months  last_login_days_ago
min                     0.5                0                 0              1                    0
max                    25.0                8                 5             36                   60

Correlación tenure_months vs meses desde signup_date: -0.014


**Interpretación:**
- `signup_date` está como texto y se puede convertir a fecha sin errores.
- `churn` es Yes/No y está desbalanceado hacia "Yes" (~57 %), algo poco realista en un negocio real.
- `monthly_fee` depende exactamente de `plan_type` (Basic=199, Standard=399, Premium=699): es redundante.
- La correlación entre `tenure_months` y la antigüedad calculada desde `signup_date` es prácticamente cero, lo que sugiere que las variables se generaron de forma independiente (dato sintético).

### 4.3 Dataset Telco: tipos, categorías y consistencia

In [25]:
print("customer_id único:", df_telco["customer_id"].is_unique)

print("\nDistribución de churn:")
print(df_telco["churn"].value_counts())
print(df_telco["churn"].value_counts(normalize=True).round(3))

print("\nÁreas:"); print(df_telco["area_code"].value_counts())
print("Estados distintos:", df_telco["state"].nunique())

# Registros con minutos diurnos en cero
print("\nRegistros con total_day_minutes = 0:", (df_telco["total_day_minutes"] == 0).sum())
display(df_telco[df_telco["total_day_minutes"] == 0])

# Cargo por minuto (debería ser casi constante si el cargo es una función de los minutos)
for franja in ["day", "eve", "night", "intl"]:
    minutos = df_telco[f"total_{franja}_minutes"]
    cargo = df_telco[f"total_{franja}_charge"]
    ratio = (cargo / minutos.where(minutos > 0))
    print(f"{franja:>5}: cargo/minuto min={ratio.min():.4f} max={ratio.max():.4f}")

# Buzón de voz
print("\nSin plan de buzón pero con mensajes:",
      ((~df_telco["voice_mail_plan"]) & (df_telco["number_vmail_messages"] > 0)).sum())
print("Con plan de buzón y 0 mensajes:",
      ((df_telco["voice_mail_plan"]) & (df_telco["number_vmail_messages"] == 0)).sum())

print("\nMáximos de variables de uso y soporte:")
print(df_telco[["account_length", "number_customer_service_calls"]].agg(["min", "max"]))

customer_id único: True

Distribución de churn:
churn
False    3652
True      598
Name: count, dtype: int64
churn
False    0.859
True     0.141
Name: proportion, dtype: float64

Áreas:
area_code
area_code_415    2108
area_code_408    1086
area_code_510    1056
Name: count, dtype: int64
Estados distintos: 51

Registros con total_day_minutes = 0: 2


,customer_id,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls,churn
341,55058c62-ff1a-4cff-aecd-fd2d2e0096d5,VT,101,area_code_510,False,False,0,0.0,0,0.0,192.1,119,16.33,168.8,95,7.60,7.2,4,1.94,1,False
1538,308d9375-7391-40be-9958-b7bb59c42a38,SD,98,area_code_415,False,False,0,0.0,0,0.0,159.6,130,13.57,167.1,88,7.52,6.8,1,1.84,4,True


  day: cargo/minuto min=0.1692 max=0.1705
  eve: cargo/minuto min=0.0849 max=0.0852
night: cargo/minuto min=0.0448 max=0.0451
 intl: cargo/minuto min=0.2682 max=0.2750

Sin plan de buzón pero con mensajes: 0
Con plan de buzón y 0 mensajes: 1

Máximos de variables de uso y soporte:
     account_length  number_customer_service_calls
min               1                              0
max             243                              9


**Interpretación:**
- `churn` es booleano y está desbalanceado (~14 % True), lo que exige métricas como F1 o recall.
- Los cargos son prácticamente una tarifa fija por minuto en cada franja, por lo que cada par minutos/cargo es redundante.
- Hay 2 registros con 0 minutos y 0 llamadas diurnas pero con uso vespertino y nocturno normal: no parecen clientes inactivos, sino posibles datos faltantes registrados como cero.
- Hay solo 3 códigos de área para 51 estados, por lo que `area_code` aporta poca información geográfica.
- `customer_id` es un identificador y no debe usarse como variable predictora.

### 4.4 Tabla de problemas identificados

In [26]:
problemas = pd.DataFrame([
    ["Suscripción", "Fechas como texto", "signup_date es object/str", "No permite cálculos temporales", "Convertir con pd.to_datetime"],
    ["Suscripción", "Variable objetivo como texto", "churn = Yes/No", "Los modelos requieren 0/1", "Mapear Yes→1, No→0"],
    ["Suscripción", "Churn muy desbalanceado hacia 'Yes'", "≈57 % Yes", "Distribución poco realista; sesgo al comparar con Telco", "Documentar y usar métricas adecuadas"],
    ["Suscripción", "tenure_months no coincide con signup_date", "Correlación ≈ 0", "Análisis temporal poco fiable", "Verificar y priorizar una de las dos variables"],
    ["Suscripción", "monthly_fee redundante con plan_type", "Un valor de tarifa por plan", "Colinealidad", "Conservar una de las dos"],
    ["Suscripción", "Datos sintéticos con rangos uniformes", "Rangos cerrados en uso, tickets y fallos", "Pueden no reflejar clientes reales", "Declarar la limitación"],
    ["Suscripción", "No hay variables de los primeros 30 días", "Solo existe tenure_months", "La pregunta analítica no se responde completamente", "Ajustar el alcance de la pregunta"],
    ["Telco", "Churn desbalanceado", "≈14 % True", "Un modelo ingenuo acierta ≈86 %", "Estratificar y usar F1/recall"],
    ["Telco", "Cargos redundantes con minutos", "cargo/minuto casi constante por franja", "Multicolinealidad", "Elegir minutos o cargos por franja"],
    ["Telco", "Minutos diurnos en cero", "2 registros con total_day_minutes = 0", "Posible dato faltante disfrazado de cero (sí tienen uso en otras franjas)", "Revisar esos registros"],
    ["Telco", "area_code poco informativo", "3 códigos para 51 estados", "Poco valor geográfico", "Tratar como categoría o descartar"],
    ["Telco", "Identificador no predictor", "customer_id es UUID", "Ruido para el modelo", "Excluir del modelado"],
    ["Ambos", "No comparten llave", "user_id vs customer_id", "No es posible unirlos", "Analizarlos por separado"],
    ["Ambos", "Etiquetas de churn distintas", "Yes/No vs True/False", "Inconsistencia al compararlos", "Normalizar a 0/1"],
], columns=["Dataset", "Problema identificado", "Evidencia encontrada", "Posible impacto", "Acción propuesta"])
problemas

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,Suscripción,Fechas como texto,signup_date es object/str,No permite cálculos temporales,Convertir con pd.to_datetime
1,Suscripción,Variable objetivo como texto,churn = Yes/No,Los modelos requieren 0/1,"Mapear Yes→1, No→0"
2,Suscripción,Churn muy desbalanceado hacia 'Yes',≈57 % Yes,Distribución poco realista; sesgo al comparar ...,Documentar y usar métricas adecuadas
3,Suscripción,tenure_months no coincide con signup_date,Correlación ≈ 0,Análisis temporal poco fiable,Verificar y priorizar una de las dos variables
4,Suscripción,monthly_fee redundante con plan_type,Un valor de tarifa por plan,Colinealidad,Conservar una de las dos
5,Suscripción,Datos sintéticos con rangos uniformes,"Rangos cerrados en uso, tickets y fallos",Pueden no reflejar clientes reales,Declarar la limitación
6,Suscripción,No hay variables de los primeros 30 días,Solo existe tenure_months,La pregunta analítica no se responde completam...,Ajustar el alcance de la pregunta
7,Telco,Churn desbalanceado,≈14 % True,Un modelo ingenuo acierta ≈86 %,Estratificar y usar F1/recall
8,Telco,Cargos redundantes con minutos,cargo/minuto casi constante por franja,Multicolinealidad,Elegir minutos o cargos por franja
9,Telco,Minutos diurnos en cero,2 registros con total_day_minutes = 0,Posible dato faltante disfrazado de cero (sí t...,Revisar esos registros


## Conclusiones

- Ambos datasets **se cargan correctamente**, no tienen valores nulos ni duplicados y sus rangos numéricos son plausibles.
- Los problemas son de **tipo de dato, redundancia entre variables, desbalance de clases y limitaciones de origen** (el dataset de suscripción es sintético).
- Antes de modelar hay que convertir fechas y etiquetas, decidir qué variables redundantes conservar y definir métricas adecuadas para clases desbalanceadas.
- La pregunta analítica (primeros 30 días) **no puede responderse con exactitud** con estos datos, porque ninguno trae información por ventana de 30 días; se recomienda ajustar el alcance.
- Los datos son **aptos para continuar** con la fase de preparación.